# Gold Analytics — Insurance Lakehouse

Build business-ready analytical datasets from validated Silver data using reusable modules in `src/`.

**Local execution:** Spark DataFrames are processed in memory. This notebook does not write Parquet, Delta, or managed tables. Persisted Delta Lake layers are a target architecture for Databricks.

In [ ]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession, functions as F

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "src" / "gold.py").exists():
    raise FileNotFoundError(f"Project root not found: {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.bronze import load_bronze_datasets
from src.silver import build_silver_datasets
from src.gold import build_gold_datasets

spark = SparkSession.builder.appName("InsuranceLakehouseGoldAnalytics").master("local[*]").getOrCreate()
print(f"Project root: {project_root}")

In [ ]:
# Reuse the tested Bronze and Silver modules; no %run dependency.
bronze_dfs = load_bronze_datasets(spark, project_root)
silver_valid_dfs, silver_rejected_dfs = build_silver_datasets(bronze_dfs, project_root)

expected_silver_counts = {"customers": 8, "policies": 8, "claims": 7}
required_columns = {
    "customers": {"customer_id", "first_name", "last_name", "province"},
    "policies": {"policy_id", "customer_id", "product", "premium", "status"},
    "claims": {"claim_id", "policy_id", "estimated_amount", "paid_amount", "status"},
}

print("\nGOLD INPUT VALIDATION")
print("-" * 65)
for name, expected in expected_silver_counts.items():
    if name not in silver_valid_dfs:
        raise ValueError(f"Missing validated Silver dataset: {name}")
    missing = required_columns[name] - set(silver_valid_dfs[name].columns)
    if missing:
        raise ValueError(f"{name}: missing required columns {sorted(missing)}")
    actual = silver_valid_dfs[name].count()
    if actual != expected:
        raise AssertionError(f"Silver {name}: expected {expected}, got {actual}")
    print(f"PASS | {name:<10} | valid records={actual}")

In [ ]:
# All Gold business logic lives in src/gold.py.
gold_dfs = build_gold_datasets(silver_valid_dfs)

expected_gold_counts = {
    "gold_policy_portfolio": 4,
    "gold_claims_summary": 3,
    "gold_customer_360": 8,
}

print("\nGOLD ANALYTICS SUMMARY")
print("-" * 65)
for name, expected in expected_gold_counts.items():
    if name not in gold_dfs:
        raise ValueError(f"Missing Gold dataset: {name}")
    actual = gold_dfs[name].count()
    if actual != expected:
        raise AssertionError(f"Gold {name}: expected {expected}, got {actual}")
    print(f"PASS | {name:<24} | records={actual}")

In [ ]:
# Reconcile Gold outputs with the validated Silver source of truth.
def metric_total(df, column):
    return df.agg(F.sum(column)).first()[0]

customers = silver_valid_dfs["customers"]
policies = silver_valid_dfs["policies"]
claims = silver_valid_dfs["claims"]
portfolio = gold_dfs["gold_policy_portfolio"]
claims_summary = gold_dfs["gold_claims_summary"]
customer_360 = gold_dfs["gold_customer_360"]

checks = [
    ("Customer count", customer_360.count(), customers.count()),
    ("Policy count", metric_total(portfolio, "policy_count"), policies.count()),
    ("Claim count", metric_total(claims_summary, "claim_count"), claims.count()),
    ("Policy premium total", metric_total(portfolio, "total_premium"), metric_total(policies, "premium")),
    ("Claim estimated amount", metric_total(claims_summary, "total_estimated_amount"), metric_total(claims, "estimated_amount")),
    ("Claim paid amount", metric_total(claims_summary, "total_paid_amount"), metric_total(claims, "paid_amount")),
    ("Customer 360 premium total", metric_total(customer_360, "total_premium"), metric_total(policies, "premium")),
    ("Customer 360 claim count", metric_total(customer_360, "claim_count"), claims.count()),
]

print("\nGOLD VALIDATION RESULTS")
print("-" * 85)
failed = 0
for name, actual, expected in checks:
    passed = actual == expected
    failed += int(not passed)
    print(f"{'PASS' if passed else 'FAIL'} | {name} | actual={actual} | expected={expected}")

print("\nGOLD VALIDATION SUMMARY")
print("-" * 85)
print(f"Total checks : {len(checks)}")
print(f"Passed       : {len(checks) - failed}")
print(f"Failed       : {failed}")
if failed:
    raise AssertionError(f"Gold validation failed: {failed} checks did not pass")
print("Gold validation completed successfully.")

### Optional data exploration

The operational flow above avoids displaying rows. To inspect a sample manually, run the cell below after uncommenting it.

In [ ]:
# Optional exploration (disabled by default):
# for name, df in gold_dfs.items():
#     print(name)
#     df.show(10, truncate=False)